# GRPO Training with CodeArena RL Benchmark

This notebook demonstrates how to connect our custom `codearena-rl-benchmark` environment to train an LLM for autonomous code debugging.
It uses the CodeArena OpenEnv server for execution feedback and reward computation.

**Run this notebook on Google Colab (Linux).**

In [ ]:
# Install Dependencies
!pip install -q trl transformers requests datasets
!pip install -q "trl[grpo]"
print("Installation complete")

In [ ]:
# Connect to CodeArena Environment
import requests
import os

# If running locally with ngrok, set this URL
# If running on HuggingFace Spaces, use your Space URL
BASE_URL = os.environ.get(
    "CODEARENA_URL",
    "https://adityanaikhpt-codearena.hf.space"
)

# Test connection
try:
    health = requests.get(f"{BASE_URL}/", timeout=10)
    print(f"Environment connected: {health.status_code}")
    obs = requests.post(
        f"{BASE_URL}/reset",
        json={"task_id": "easy"},
        timeout=10
    ).json()
    print(f"Reset successful. Keys: {list(obs.keys())}")
    print(f"Curriculum: {obs.get('curriculum_info', {})}")
except Exception as e:
    print(f"Connection failed: {e}")
    print("Make sure the server is running and BASE_URL is correct")

In [ ]:
# Define Reward Function and Training Loop
import json
import time
import requests
import matplotlib.pyplot as plt

# Store rewards across training steps
step_rewards = []
episode_rewards = []
step_count = 0
NUM_EPISODES = 20

def get_fix_from_model(observation: dict) -> str:
    """
    In a real GRPO setup this would be your policy model.
    Here we use the same OpenAI-compatible inference used in
    inference.py to demonstrate the training loop.
    """
    import os
    from openai import OpenAI

    client = OpenAI(
        base_url=os.environ.get("API_BASE_URL", ""),
        api_key=os.environ.get("HF_TOKEN", "NO_KEY")
    )

    buggy_code = observation.get("buggy_code", "")
    error_log = observation.get("error_log", "")
    test_results = observation.get("test_results", "")

    response = client.chat.completions.create(
        model=os.environ.get("MODEL_NAME", "Qwen/Qwen2.5-72B-Instruct"),
        max_tokens=1024,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an expert Python debugger. "
                    "Return ONLY the fixed Python code. "
                    "No explanation, no markdown, no backticks."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Fix this broken Python code:\n\n{buggy_code}\n\n"
                    f"Error log:\n{error_log}\n\n"
                    f"Test results:\n{test_results}\n\n"
                    f"Return ONLY the fixed Python code."
                )
            }
        ]
    )
    return response.choices[0].message.content.strip()

# Training loop
print(f"Starting {NUM_EPISODES} episodes...\n")

for episode in range(NUM_EPISODES):
    # Reset environment
    reset_resp = requests.post(
        f"{BASE_URL}/reset",
        json={"task_id": "auto"},
        timeout=15
    ).json()

    difficulty = reset_resp.get(
        "curriculum_info", {}
    ).get("current_difficulty", "unknown")
    episode_reward_sum = 0
    episode_steps = 0

    # Run up to 5 steps per episode
    for step in range(5):
        observation = reset_resp if step == 0 else step_resp.get("observation", {})

        # Get fix from model
        try:
            proposed_fix = get_fix_from_model(observation)
        except Exception as e:
            print(f"  Model call failed: {e}")
            break

        # Submit fix to environment
        step_resp = requests.post(
            f"{BASE_URL}/step",
            json={"proposed_fix": proposed_fix},
            timeout=30
        ).json()

        reward = step_resp.get("reward", 0.0)
        done = step_resp.get("done", False)
        components = step_resp.get("reward_components", {})

        step_rewards.append(reward)
        episode_reward_sum += reward
        episode_steps += 1
        step_count += 1

        print(
            f"Episode {episode+1:02d} | Step {step+1} | "
            f"Difficulty: {difficulty} | "
            f"Reward: {reward:.3f} | "
            f"Tests: {components.get('test_pass_ratio', 0):.2f} | "
            f"LLM: {components.get('llm_correctness', 0):.2f}"
        )

        if done:
            break

    avg = episode_reward_sum / max(episode_steps, 1)
    episode_rewards.append(avg)
    print(f"Episode {episode+1:02d} complete. Avg reward: {avg:.3f}\n")
    time.sleep(1)

print(f"Training complete. Total steps: {step_count}")

In [ ]:
# Plot Reward Curves
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Step rewards with rolling average
ax1.plot(step_rewards, alpha=0.4, color="#4C72B0", label="Step reward")
if len(step_rewards) >= 10:
    rolling = np.convolve(
        step_rewards,
        np.ones(10)/10,
        mode='valid'
    )
    ax1.plot(
        range(9, len(step_rewards)),
        rolling,
        color="#DD8452",
        linewidth=2,
        label="10-step rolling avg"
    )
ax1.set_xlabel("Training Step")
ax1.set_ylabel("Reward (0-1)")
ax1.set_ylim(0, 1)
ax1.set_title("CodeArena: Reward Over Training Steps")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Episode average rewards
ax2.plot(
    episode_rewards,
    marker='o',
    color="#55A868",
    linewidth=2,
    markersize=6,
    label="Episode avg reward"
)
ax2.set_xlabel("Episode")
ax2.set_ylabel("Average Reward (0-1)")
ax2.set_ylim(0, 1)
ax2.set_title("CodeArena: Average Reward Per Episode")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_results.png", dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved as training_results.png")

# Print summary stats
print(f"\nTraining Summary:")
print(f"  Total steps:       {len(step_rewards)}")
print(f"  Total episodes:    {len(episode_rewards)}")
print(f"  Best step reward:  {max(step_rewards):.3f}")
print(f"  Final avg reward:  {episode_rewards[-1]:.3f}")
print(f"  Improvement:       "
      f"{episode_rewards[-1] - episode_rewards[0]:+.3f}")

In [ ]:
# Environment Info
# Show environment details for judges
import requests

curriculum = requests.get(f"{BASE_URL}/curriculum").json()
print("Current Curriculum State:")
print(f"  Difficulty:     {curriculum['current_difficulty']}")
print(f"  Avg Reward:     {curriculum['recent_avg_reward']}")
print(f"  Episodes Seen:  {curriculum['episodes_tracked']}")

print("\nEnvironment Endpoints:")
print(f"  POST {BASE_URL}/reset")
print(f"  POST {BASE_URL}/step")
print(f"  GET  {BASE_URL}/state")
print(f"  GET  {BASE_URL}/curriculum")

print("\nReward Function:")
print("  compile_score    \u00d7 0.20")
print("  test_pass_ratio  \u00d7 0.40")
print("  efficiency_score \u00d7 0.10")
print("  llm_judge_score  \u00d7 0.30")
print("  step_penalty     \u2212 0.02 per step")
print("  novelty_penalty  \u2212 0.10 if repeated fix")
print("  clamped to       [0.001, 0.999]")